# ACC 380K — Homework 4

## One function from firms and years to a chart

**Assignment version 0.6.7 · October 26, 2026**

In Homework 3 you were given three accession numbers and counted a phrase in each filing. In this
homework you build the whole pipeline yourself, as **one function** that works for any companies,
any range of years, any search phrase, and any filing type.

### Your goal

Write a function with this signature (keep these names and this argument order):

```python
term_counts(first_year, last_year, ciks, term, sec_identity, form_type="10-K")
```

- `first_year`, `last_year`: integers; calendar years of the **filing date**. Include every filing
  filed from January 1 of `first_year` through December 31 of `last_year`. (This is not the company's
  fiscal year: a company's fiscal-2018 10-K may be filed in 2018 or 2019.)
- `ciks`: a **list** of company CIK numbers (integers), e.g. `[829224, 63908]`.
- `term`: the search phrase, e.g. `"China"`. Count exact, case-sensitive matches.
- `sec_identity`: the name and email of **the person running the function** (for example,
  `"Jane Doe jane.doe@utexas.edu"`), sent to the SEC with every request. Do not hard-code your own
  identity inside the function: whoever calls it supplies theirs. When we grade, we will pass our own.
  Your function must use this argument both for `edgar.set_identity` and for the `User-Agent`
  header on any `requests.get` call.
- `form_type`: the filing type to keep; default `"10-K"`. Your function must also work for other types such as `"10-Q"`. Keep only rows whose form type matches exactly (so `"10-K/A"` amendments are left out).

The function must do two things:

1. **Return a DataFrame** with **one row per filing** and exactly these columns, in this order:

   | column | contents |
   |---|---|
   | `cik` | the company's CIK (integer) |
   | `company` | the company name as it appears in the EDGAR index |
   | `filing_date` | the date filed, as `"YYYY-MM-DD"` |
   | `accession` | the accession number, e.g. `"0000829224-25-000114"` |
   | `count` | the number of times `term` appears in the filing's text |

2. **Display a line plot** of `count` over time, **one line per company**, with company names in
   the legend. Put `filing_date` on the x-axis (as dates) for every form type.

How you build it is up to you. You may break it into smaller helper functions, reuse your
Homework 3 code, or have AI write it from scratch (see "Working with AI" below). Optional hints and a
suggested roadmap follow below.

### Working with AI

You are encouraged to use AI (for example, Gemini in Colab) to write this code. You are responsible
for it: **read every line, understand what it does, and confirm it gives correct answers.** The
written question at the end asks you to explain your code.

**If you don't understand the AI's code, you have two options.** Don't leave a line in your notebook that you can't explain.

1. **Ask the AI to teach you.** Have it explain the code line by line until you could explain it yourself. For example: "Explain what this line does, step by step, as if I've only learned loops and pandas masks."
2. **Ask the AI to rewrite it with tools you know.** Work with it to replace the code with an approach you already understand. For example: "Rewrite this using only a for loop and pd.concat, which I already know."

**Steer the tools.** Left alone, AI often picks tools we haven't covered. You can ask it to use
tools you already know, so you are better equipped to audit what it writes. For example, add lines
like these to your prompt:

- "Use `pd.read_csv` and boolean masks to filter the index, and `pd.concat` to combine tables."
- "Use for loops, not while loops or list comprehensions."
- "Don't hide errors: if a download fails, raise or print it clearly rather than recording a count of 0."

**Two ways to audit.** *Auditing the code* means reading each line and knowing what it does.
*Auditing the output* means checking the results for problems: do the 2018 counts match the check
values, is the row count right, are there all zeros, missing years, or wrong columns? Output
auditing is often faster, but it is riskier, and it works best when you already know what good
output should look like. Here, use both: read the code, and check the output against the known or plausible values.

**Test beyond the example.** When we grade, we will run your function with **different arguments**:
other companies, other year ranges, other search terms, and other form types such as `"10-Q"`. Test
it yourself on several combinations, not just Starbucks, McDonald's, and "China". For example:

```python
term_counts(2019, 2023, [320193, 789019], "supply chain", sec_identity="Your Name your.email@utexas.edu")     # Apple, Microsoft; use your own name and email
term_counts(2023, 2024, [21344], "inflation", sec_identity="Your Name your.email@utexas.edu", form_type="10-Q")  # Coca-Cola; use your own name and email
```

### Before you begin

1. Choose **File → Save a copy in Drive** and name your copy `lastname_firstname_homework04.ipynb`.
2. Test on a small case (one company, one or two years) before running the full test.
3. Before submitting, restart and run your completed notebook in order. Download the `.ipynb`
   file and submit it as directed in Canvas.

## Optional hints

You do not have to follow these. They are here if you get stuck.

- **Installing EdgarTools.** Colab needs `!pip install edgartools==5.58.0` once per session. Use this
  exact version: newer versions convert filings to text slightly differently, so counts can change
  (Starbucks 2018 is 105 instead of 103 on version 5.62.1).
- **Imports.** Put every `import` your function needs (for example `import pandas as pd`) inside the
  function, so it runs on its own when we call it. The `!pip install` line stays in its own cell; it
  is not Python and cannot go inside the function.
- **Progress messages.** Downloads are slow, so it helps to `print` a short status message as the
  function works, e.g. `"Downloading the index file for Q4 2022"` or `"Counting filing 5 of 32"`.

- **Where to find the filings.** The SEC publishes a quarterly index of every filing at
  `https://www.sec.gov/Archives/edgar/full-index/{year}/QTR{quarter}/master.idx`
  (you downloaded and read these in Notebook 8). Each row has the CIK, company name, form type,
  date filed, and a filename. The accession number is the last part of the filename, without `.txt`.
  Send `sec_identity` as the `User-Agent` header, and add `timeout=60` to `requests.get` so a
  stalled download stops with an error instead of hanging forever.
- **Don't skip failed downloads.** If an index download does not succeed (`status_code` is not 200),
  raise an error rather than skipping that quarter. Otherwise filings quietly go missing.
- **Identity for EdgarTools.** Call `edgar.set_identity(sec_identity)` inside your function, before
  downloading any filing.
- **Searching and counting.** Homework 3 already did this part: given a filing's text, count how
  many times the phrase appears.
- **Downloading a filing's text.** With EdgarTools, these two lines download one filing as Markdown text:

  ```python
  filing = edgar.get_by_accession_number("0000829224-25-000114")
  text = filing.document.markdown()
  ```
- **Test small first.** Every filing is a separate download, and the index files are large. Try one
  company and one or two years before running all of 2010–2025.

### A suggested roadmap (plain English)

1. For each year and quarter in the range, download the index file and keep the rows for your
   companies and form type. Combine them into one table.
2. For each kept row, get the accession number and download that filing's text.
3. Count the search phrase in the text and record one row: CIK, company, filing date, accession, count.
4. Turn the rows into a DataFrame, plot one line per company, and return the DataFrame.

## Your function

Write `term_counts` in the cell below. You may add more cells above it for helper functions.

In [ ]:
def term_counts(first_year, last_year, ciks, term, sec_identity, form_type="10-K"):
    """Count term in each matching filing, plot the counts, and return one row per filing.

    first_year, last_year: calendar years of the filing date (inclusive).
    sec_identity: name and email of the person running the function, e.g. "Jane Doe jane@utexas.edu".
    """
    # Your code here

### Test small first

Run your function on a small case. Change it as you like while testing.

In [ ]:
term_counts(2018, 2018, [829224], "China", sec_identity="Your Name your.email@utexas.edu")  # put your own name and email here

### Full test

Run the full test: Starbucks (`829224`) and McDonald's (`63908`), 2010 to 2025, 10-Ks, searching for
`"China"`. You should get 32 rows (one 10-K per company per year).

**Check your results:** for the 10-Ks filed in 2018, the counts should be **103** for Starbucks
(filed 2018-11-16, its fiscal-2018 10-K) and **29** for McDonald's (filed 2018-02-23, its fiscal-2017 10-K).
These values assume EdgarTools 5.58.0 (see the hints). If yours differ, find out why before you submit.

In [ ]:
china_counts = term_counts(2010, 2025, [829224, 63908], "China", sec_identity="Your Name your.email@utexas.edu")  # put your own name and email here
china_counts

### Another form type

Show that your function also works for 10-Qs. Keep this small: two companies, one year. You should get 6 rows (3 per company; Q4 is covered by the 10-K).

In [ ]:
term_counts(2024, 2024, [829224, 63908], "China", sec_identity="Your Name your.email@utexas.edu", form_type="10-Q")  # put your own name and email here

## Written question

In a few sentences: What does each part of your code do? How did you confirm that it works
(for example, which counts did you check, and how)?

*Your answer here.*

## Before submitting

- Your notebook runs top to bottom without errors after a restart.
- `term_counts` returns a DataFrame with exactly the columns `cik`, `company`, `filing_date`,
  `accession`, `count`, one row per filing.
- The full test shows 32 rows and a chart with one line per company, labeled with company names.
- Your 2018 counts match the check values above (with edgartools 5.58.0).
- The 10-Q test runs.
- You tested `term_counts` on other companies, years, search terms, and form types (grading
  will use different arguments), and the output made sense each time.
- You answered the written question, and you can explain every line of your code, including any
  code AI wrote for you.

If the SEC connection remains unavailable, tell the instructor or TA rather than inventing
filing text or counts.